In [ ]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys, math, warnings
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/spikeparam')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/shared_helper_modules')

from ridge_regression_utils import (
    load_cell_data, build_ridge_matrices,
    run_ridge_regression, apply_fdr, plot_ridge_results,
    WAVEFORM_LABELS,
)

warnings.filterwarnings('ignore', message='use_inf_as_na option is deprecated',
                        category=FutureWarning, module='seaborn')
warnings.filterwarnings('ignore', message=r'invalid value encountered in log10',
                        category=RuntimeWarning, module=r'.*specparam.*')

In [ ]:
# ── Cell configuration ────────────────────────────────────────────────────────
cell_num = 28

# Symmetric 50 ms windows immediately around the spike
PRE_WIN      = (-0.055, -0.005)  # pre-spike:  −55 → −5 ms
POST_WIN     = ( 0.005,  0.055)  # post-spike: +5 → +55 ms
BASELINE_WIN = (-0.20,  -0.10)   # baseline: −200 → −100 ms

# Ridge regression settings
ALPHAS   = np.logspace(-3, 3, 100)
N_PERM   = 1000
RNG_SEED = 42

In [ ]:
# ── 1. Load data ──────────────────────────────────────────────────────────────
df_reg, specparam_by_spike, lfp_windows_by_spike = load_cell_data(cell_num)

In [ ]:
# ── 2. Build matrices ─────────────────────────────────────────────────────────
(X_waveform, X_log_isi, X_both, waveform_labels,
 Y, target_names, target_labels) = build_ridge_matrices(
    df_reg, specparam_by_spike, lfp_windows_by_spike,
    pre_win=PRE_WIN, post_win=POST_WIN, baseline_win=BASELINE_WIN,
)

predictor_sets = {
    'Waveform only':      (X_waveform, waveform_labels),
    'Log ISI only':       (X_log_isi,  ['Log ISI']),
    'Waveform + Log ISI': (X_both,     waveform_labels + ['Log ISI']),
}

In [ ]:
# ── 3. Ridge regression (5-fold CV + permutation test) ────────────────────────
results = run_ridge_regression(
    Y, predictor_sets, target_names,
    n_perm=N_PERM, rng_seed=RNG_SEED, alphas=ALPHAS,
)

In [ ]:
# ── 4. FDR correction + summary table ─────────────────────────────────────────
results = apply_fdr(results, target_names, predictor_sets)

In [ ]:
# ── 5. Plots ──────────────────────────────────────────────────────────────────
plot_ridge_results(
    cell_num, results, target_names, target_labels,
    predictor_sets, waveform_labels, Y, X_waveform,
)